# Java for Competitive Programming: C++ Cheatsheet

Everything from our chat in one place: what the example solutions have in common, a starter template, a C++ to Java translation table, gotchas, and input/output patterns.

All cells are Markdown with Java code blocks, so nothing here needs a Java kernel. Copy the snippets into your editor.

**Contents**
1. What the example solutions have in common
2. Starter template
3. Initialising arrays and lists (the `vector<int> v(10, 9)` question)
4. C++ to Java tables
5. Gotchas for C++ people
6. Input: reusable reader
7. Input: common formats
8. Input: graphs
9. Input: `BufferedReader` fallback
10. Output
11. Which reader to pick

## 1. What the example solutions have in common

They all solve the same problem: find the fewest-edge path, break ties by the smallest digit string, and print that string mod 998244353.

**Idea level**
- Comparing digit strings directly is too slow. Every solution avoids it by grouping vertices that share the same string, either with a "rank" per vertex (Dijkstra-style) or by processing whole groups at once (BFS layers).
- The mod value is computed incrementally as `(prev * 10 + digit) % MOD`.

**Java habits**
- `public class Main`: judges like AtCoder require this name.
- Custom fast input: nobody uses `Scanner`, which is very slow. They use a byte-buffer reader or `BufferedReader` plus `split`/`StringTokenizer`.
- Fast output: `PrintWriter` or `StringBuilder`, flushed once at the end.
- Adjacency lists: `List<List<X>>`, `ArrayList<int[]>[]` (with an unchecked-generic warning), or a hand-written `IntArrayList` to avoid boxing.
- Small `static class`es (`Edge`, `Node`, `Path`) implementing `Comparable` so they work in `PriorityQueue`.
- `long` for mod arithmetic, `static final` constants instead of `#define`.
- `Arrays.fill(arr, -1)` for sentinel values, and `-1` on input to make vertices 0-indexed.
- Wildcard imports (`java.util.*`) at the top.
- One solution was machine-generated by a tool that merges a personal library into one file, which is why everything is fully qualified (`java.lang.Integer`). Its `solve` is recursive, which risks stack overflow on deep graphs in Java.

## 2. Starter template

```java
import java.io.*;
import java.util.*;

public class Main {
    static final int MOD = 998244353;
    static DataInputStream in = new DataInputStream(new BufferedInputStream(System.in, 1 << 16));

    static int ni() throws IOException {
        int r = 0, b = in.read();
        while (b < '0') b = in.read();           // skip whitespace (ignores negatives)
        while (b >= '0') { r = r * 10 + b - '0'; b = in.read(); }
        return r;
    }

    public static void main(String[] args) throws IOException {
        PrintWriter out = new PrintWriter(new BufferedWriter(new OutputStreamWriter(System.out)));
        int n = ni();
        // ... solve ...
        out.flush();
    }
}
```

For deep recursion, run the work in a thread with a big stack:

```java
new Thread(null, () -> { /* solve */ }, "", 1L << 28).start();
```

## 3. Initialising arrays and lists

**Why `private ArrayList<Integer> bonuses(10,0);` does not work.** Java's `ArrayList` has no constructor taking a size and a fill value. `new ArrayList<>(10)` sets only the *capacity* (a hint), and the list still has 0 elements. Also, constructor arguments can't go after the variable name; fields need `= new ...`.

**10 zeros in a list**

```java
private List<Integer> bonuses = new ArrayList<>(Collections.nCopies(10, 0));
```

`Collections.nCopies` is immutable, so wrapping it in `new ArrayList<>(...)` gives a mutable copy.

**If the size is fixed, use an array** (the real analogue of a sized `vector`):

```java
private int[] bonuses = new int[10];   // auto-initialised to 0
```

**Fill with something other than 0**

```java
int[] a = new int[10];
Arrays.fill(a, 9);                       // all 9s (like vector<int> v(10, 9))
Arrays.fill(a, from, to, 9);             // range form, `to` is exclusive

int[] b = {9, 9, 9, 9, 9};               // literal, fine for small arrays
int[] c = IntStream.generate(() -> 9).limit(10).toArray();   // streams

int[] squares = new int[10];
Arrays.setAll(squares, i -> i * i);      // value depends on index
```

As a field, wrap the statement in an instance initialiser block:

```java
private int[] bonuses = new int[10];
{
    Arrays.fill(bonuses, 9);
}
```

**2D arrays:** `Arrays.fill` on the outer array makes every row point to the *same* inner array. Fill each row:

```java
int[][] grid = new int[5][10];
for (int[] row : grid) Arrays.fill(row, 9);
```

**Gotcha:** `new ArrayList<>(10)` then `list.set(3, 5)` throws `IndexOutOfBoundsException`, because capacity is not size.

## 4. C++ to Java tables

Every row shows a concrete example in both columns, so you can copy either side as is.

### Basics and types

| C++ | Java |
|---|---|
| `long long x = 1000000000000000000LL;` | `long x = 1_000_000_000_000_000_000L;` |
| `bool ok = true;` | `boolean ok = true;` |
| `unsigned x = 4000000000u; cout << x;` | no unsigned types: `int x = (int) 4000000000L; out.println(Integer.toUnsignedString(x));` and `Long.compareUnsigned(a, b)` |
| `__int128 p = (__int128) a * b;` | `BigInteger p = BigInteger.valueOf(a).multiply(BigInteger.valueOf(b));` |
| `auto n = v.size();` | `var n = list.size();` |
| `#define N 200005` / `const int MOD = 998244353;` | `static final int N = 200005;` / `static final int MOD = 998244353;` |
| `typedef long long ll; ll x = 5;` | no `typedef`: write `long x = 5;` everywhere |
| `Vec c = a + b;` (operator overloading) | no overloading: `Vec c = a.add(b);` |
| `int inf = INT_MAX; long long linf = LLONG_MAX;` | `int inf = Integer.MAX_VALUE; long linf = Long.MAX_VALUE;` |
| `long long m = 1LL << 40;` | `long m = 1L << 40;` |
| `unsigned y = (unsigned) x >> 1;` (logical shift) | `int y = x >>> 1;` (`x >> 1` is arithmetic) |
| `swap(v[i], v[j]);` | `Collections.swap(list, i, j);` or arrays: `int t = a[i]; a[i] = a[j]; a[j] = t;` |

### Arrays and vectors

| C++ | Java |
|---|---|
| `vector<int> v(n, x);` | `int[] v = new int[n]; Arrays.fill(v, x);` |
| `vector<vector<int>> a(n, vector<int>(m));` | `int[][] a = new int[n][m];` |
| `int n = v.size(); int m = s.size();` | `int n = list.size(); int k = a.length; int m = s.length();` |
| `vector<int> g[n];` | `List<Integer>[] g = new ArrayList[n]; for (int i = 0; i < n; i++) g[i] = new ArrayList<>();` |
| `v.push_back(x);` | `list.add(x);` |
| `v[i] = x; int y = v[i];` | `list.set(i, x); int y = list.get(i);` (arrays: `a[i] = x; int y = a[i];`) |
| `v.insert(v.begin() + i, x);` | `list.add(i, x);` |
| `int b = v.back(); v.pop_back();` | `int b = list.get(list.size() - 1); list.remove(list.size() - 1);` |
| `v.erase(v.begin() + i);` | `list.remove(i);` (index) |
| `v.erase(find(v.begin(), v.end(), x));` | `list.remove(Integer.valueOf(x));` (value) |
| `bool e = v.empty(); v.clear();` | `boolean e = list.isEmpty(); list.clear();` |
| `memset(a, 0, sizeof a);` | `Arrays.fill(a, 0);` (new arrays are already zeroed) |
| `reverse(v.begin(), v.end());` | `Collections.reverse(list);` (arrays: `for (int i = 0, j = n - 1; i < j; i++, j--) { int t = a[i]; a[i] = a[j]; a[j] = t; }`) |
| `for (auto x : v) sum += x;` | `for (int x : a) sum += x;` |
| `vector<int> w = v;` | `int[] w = a.clone();` or `List<Integer> w = new ArrayList<>(list);` |
| `long long s = accumulate(v.begin(), v.end(), 0LL);` | `long s = 0; for (int x : a) s += x;` or `long s = Arrays.stream(a).asLongStream().sum();` |
| `int mx = *max_element(v.begin(), v.end());` | `int mx = Arrays.stream(a).max().getAsInt();` or `Collections.max(list)` |

### Sorting

| C++ | Java |
|---|---|
| `sort(a, a + n);` | `Arrays.sort(a);` |
| `sort(a + l, a + r);` | `Arrays.sort(a, l, r);` (`r` exclusive) |
| `sort(v.begin(), v.end());` | `Collections.sort(list);` or `list.sort(null);` |
| `sort(v.begin(), v.end(), greater<int>());` | `Integer[] a = {3, 1, 2}; Arrays.sort(a, Collections.reverseOrder());` or `list.sort(Collections.reverseOrder());` |
| `sort(v.begin(), v.end(), [](auto& x, auto& y) { return x[1] < y[1]; });` | `Arrays.sort(arr, (x, y) -> Integer.compare(x[1], y[1]));` (`arr` is `int[][]` or an object array) |
| `sort(v.begin(), v.end(), [](auto& x, auto& y) { return x[0] != y[0] ? x[0] < y[0] : x[1] > y[1]; });` (by first, then second descending) | `Arrays.sort(arr, Comparator.comparingInt((int[] q) -> q[0]).thenComparingInt(q -> -q[1]));` |
| `pair<int,int> p = {a, b}; int f = p.first, s = p.second;` | `int[] p = {a, b}; int f = p[0], s = p[1];` or `record P(int a, int b) {}` with `P p = new P(a, b); int f = p.a();` |

### Containers: stack, queue, heap

| C++ | Java |
|---|---|
| `priority_queue<int> pq;` (max-heap) | `PriorityQueue<Integer> pq = new PriorityQueue<>(Collections.reverseOrder());` |
| `priority_queue<int, vector<int>, greater<int>> pq;` (min-heap) | `PriorityQueue<Integer> pq = new PriorityQueue<>();` (default is a **min-heap**) |
| `pq.push(x); int t = pq.top(); pq.pop();` | `pq.add(x); int t = pq.peek(); pq.poll();` (`poll` returns and removes) |
| `using P = pair<long long,int>; priority_queue<P, vector<P>, greater<P>> pq; pq.push({d, v});` | `PriorityQueue<long[]> pq = new PriorityQueue<>((x, y) -> Long.compare(x[0], y[0])); pq.add(new long[]{d, v});` |
| `queue<int> q; q.push(1); int f = q.front(); q.pop();` | `ArrayDeque<Integer> q = new ArrayDeque<>(); q.add(1); int f = q.peek(); q.poll();` |
| `deque<int> d; d.push_front(1); d.push_back(2); d.pop_back();` | `ArrayDeque<Integer> d = new ArrayDeque<>(); d.addFirst(1); d.addLast(2); d.pollLast();` |
| `stack<int> st; st.push(1); int t = st.top(); st.pop();` | `ArrayDeque<Integer> st = new ArrayDeque<>(); st.push(1); int t = st.peek(); st.pop();` (avoid the `Stack` class) |
| `q.empty()` / `q.size()` | `q.isEmpty()` / `q.size()` |

### Containers: `unordered_map` / `HashMap`

| C++ | Java |
|---|---|
| `unordered_map<int,int> mp;` | `Map<Integer,Integer> mp = new HashMap<>(Map.of(2,4,8,9));` |
| `mp[5] = 3;` | `mp.put(5, 3);` |
| `int v = mp[5];` (inserts 0 if missing) | `int v = mp.get(5);` (throws `NullPointerException` on unboxing if missing) |
| `auto it = mp.find(7); int w = it == mp.end() ? 0 : it->second;` | `int w = mp.getOrDefault(7, 0);` (does not insert) |
| `mp.count(5)` or `mp.contains(5)` (C++20) | `mp.containsKey(5)` |
| `mp[k]++;` | `mp.merge(k, 1, Integer::sum);` or `mp.put(k, mp.getOrDefault(k, 0) + 1);` |
| `mp.try_emplace(9, 0);` | `mp.putIfAbsent(9, 0);` |
| `mp.erase(5);` | `mp.remove(5);` |
| `mp.size(); mp.empty(); mp.clear();` | `mp.size(); mp.isEmpty(); mp.clear();` |
| `for (auto& [k, v] : mp) { }` | `for (Map.Entry<Integer,Integer> e : mp.entrySet()) { int k = e.getKey(), v = e.getValue(); }` |
| keys only: `for (auto& [k, _] : mp) { }` | `for (int k : mp.keySet()) { }` |
| values only: `for (auto& [_, v] : mp) { }` | `for (int v : mp.values()) { }` |
| `map<int, vector<int>> adj; adj[u].push_back(v);` | `Map<Integer,List<Integer>> adj = new HashMap<>(); adj.computeIfAbsent(u, k -> new ArrayList<>()).add(v);` |
| `map<int,int> freq; for (int x : a) freq[x]++;` | `Map<Integer,Integer> freq = new HashMap<>(); for (int x : a) freq.merge(x, 1, Integer::sum);` |
| `map<Dir,int> cnt; cnt[UP]++;` (enum key) | `Map<Dir,Integer> cnt = new EnumMap<>(Dir.class); cnt.merge(Dir.UP, 1, Integer::sum);` |

`HashMap` iteration order is unspecified, same as `unordered_map`. Use `LinkedHashMap` for insertion order. On Codeforces, `HashMap<Integer, ...>` can be attacked with adversarial keys; use `TreeMap` or XOR keys with a random constant when hacks are likely.

### Containers: `unordered_set` / `HashSet` and `contains`

| C++ | Java |
|---|---|
| `unordered_set<int> st;` | `Set<Integer> st = new HashSet<>();` |
| `st.insert(4);` | `st.add(4);` |
| `bool fresh = st.insert(4).second;` | `boolean fresh = st.add(4);` (`false` if already present) |
| `st.count(4)` or `st.contains(4)` (C++20) | `st.contains(4)` |
| `st.erase(4);` | `st.remove(4);` |
| `st.size(); st.empty();` | `st.size(); st.isEmpty();` |
| `for (int x : st) { }` | `for (int x : st) { }` |
| `find(v.begin(), v.end(), x) != v.end()` | `list.contains(x)` (O(n)) |
| `find(a, a + n, x) != a + n` | `Arrays.stream(a).anyMatch(y -> y == x)` (arrays have no `contains`; use a `Set` if you query often) |
| `s.find("ab") != string::npos` | `s.contains("ab")` |
| `set<pair<int,int>> seen; seen.insert({0, 1}); seen.count({0, 1});` | `record P(int r, int c) {}` then `Set<P> seen = new HashSet<>(); seen.add(new P(0, 1)); seen.contains(new P(0, 1));` |

An `int[]` is **not** usable as a key, because arrays compare by reference. Use a `record` (Java 16+, declared at class level), a `String`, or a packed `long` such as `(long) r * W + c`.

### Containers: `map` / `set` (sorted) and `multiset`

| C++ | Java |
|---|---|
| `map<int,int> tm;` | `TreeMap<Integer,Integer> tm = new TreeMap<>();` |
| `int lo = tm.begin()->first; int hi = tm.rbegin()->first;` | `int lo = tm.firstKey(); int hi = tm.lastKey();` |
| `tm.lower_bound(x)->first` (first key >= x) | `tm.ceilingKey(x)` |
| `tm.upper_bound(x)->first` (first key > x) | `tm.higherKey(x)` |
| `prev(tm.upper_bound(x))->first` (last key <= x) | `tm.floorKey(x)` |
| `prev(tm.lower_bound(x))->first` (last key < x) | `tm.lowerKey(x)` |
| `tm.erase(tm.begin());` | `tm.pollFirstEntry();` |
| `set<int> ts; ts.insert(3);` | `TreeSet<Integer> ts = new TreeSet<>(); ts.add(3);` |
| `int lo = *ts.begin(); int hi = *ts.rbegin();` | `int lo = ts.first(); int hi = ts.last();` |
| `*ts.lower_bound(x)` / `*ts.upper_bound(x)` | `ts.ceiling(x)` (>= x) / `ts.higher(x)` (> x) |
| `*prev(ts.upper_bound(x))` / `*prev(ts.lower_bound(x))` | `ts.floor(x)` (<= x) / `ts.lower(x)` (< x) |
| `ts.erase(ts.begin());` | `ts.pollFirst();` |
| `multiset<int> ms; ms.insert(x); ms.erase(ms.find(x));` | `TreeMap<Integer,Integer> ms = new TreeMap<>(); ms.merge(x, 1, Integer::sum);` then to remove one: `if (ms.merge(x, -1, Integer::sum) == 0) ms.remove(x);` |
| `int i = lower_bound(a, a + n, x) - a;` | none usable (`Arrays.binarySearch` is not first-occurrence); write `int lo = 0, hi = n; while (lo < hi) { int mid = (lo + hi) >>> 1; if (a[mid] < x) lo = mid + 1; else hi = mid; }` and `lo` is the index |

The `floor`, `ceiling`, `higher` and `lower` methods return `null` when nothing matches, so check before unboxing.

### Strings

| C++ | Java |
|---|---|
| `char c = s[i];` | `char c = s.charAt(i);` |
| `string t = s.substr(i, len);` | `String t = s.substring(i, i + len);` (end exclusive) |
| `string s; for (char c : t) s += c;` | `StringBuilder sb = new StringBuilder(); for (char c : t) sb.append(c); String s = sb.toString();` |
| `bool same = (s == t);` | `boolean same = s.equals(t);` (`==` compares references) |
| `bool less = s < t;` | `boolean less = s.compareTo(t) < 0;` |
| `size_t p = s.find("ab");` | `int p = s.indexOf("ab");` (-1 if missing) |
| `s.find("ab") != string::npos` | `s.contains("ab")` |
| `s.rfind("ab", 0) == 0` (starts with) | `s.startsWith("ab")` (also `s.endsWith("ab")`) |
| `string t = to_string(x);` | `String t = String.valueOf(x);` |
| `int x = stoi(s); long long y = stoll(s);` | `int x = Integer.parseInt(s); long y = Long.parseLong(s);` |
| `s[0] = 'x';` (mutable `string`) | `char[] c = s.toCharArray(); c[0] = 'x'; s = new String(c);` |
| `sort(s.begin(), s.end());` | `char[] c = s.toCharArray(); Arrays.sort(c); s = new String(c);` |
| `reverse(s.begin(), s.end());` | `s = new StringBuilder(s).reverse().toString();` |
| `int d = s[i] - '0'; bool dig = isdigit(s[i]);` | `int d = s.charAt(i) - '0'; boolean dig = Character.isDigit(s.charAt(i));` |
| `stringstream ss(line); string w; while (ss >> w) { }` | `for (String w : line.trim().split("\\s+")) { }` or `new StringTokenizer(line)` |

### Math and bits

| C++ | Java |
|---|---|
| `int y = abs(x); int m = min(a, b); double r = sqrt(x);` | `int y = Math.abs(x); int m = Math.min(a, b); double r = Math.sqrt(x);` |
| `double p = pow(2, 10);` | `double p = Math.pow(2, 10);` (returns `double`) |
| `long long modpow(long long b, long long e, long long m) { long long r = 1; b %= m; while (e > 0) { if (e & 1) r = r * b % m; b = b * b % m; e >>= 1; } return r; }` | `static long modpow(long b, long e, long m) { long r = 1; b %= m; while (e > 0) { if ((e & 1) == 1) r = r * b % m; b = b * b % m; e >>= 1; } return r; }` |
| `long long g = __gcd(a, b);` | `static long gcd(long a, long b) { return b == 0 ? a : gcd(b, a % b); }` then `long g = gcd(a, b);` |
| `int r = ((a % m) + m) % m;` | `int r = Math.floorMod(a, m);` |
| `int c = __builtin_popcount(x); int d = __builtin_popcountll(y);` | `int c = Integer.bitCount(x); int d = Long.bitCount(y);` |
| `int lz = __builtin_clz(x); int tz = __builtin_ctz(x);` | `int lz = Integer.numberOfLeadingZeros(x); int tz = Integer.numberOfTrailingZeros(x);` |
| `do { } while (next_permutation(a, a + n));` | none; use backtracking: `void perm(int k) { if (k == n) { System.out.println(Arrays.toString(p)); return; } for (int i = 0; i < n; i++) if (!used[i]) { used[i] = true; p[k] = i; perm(k + 1); used[i] = false; } }` |

### I/O (details in sections 6 to 10)

| C++ | Java |
|---|---|
| `int x; cin >> x;` | `int x = sc.nextInt();` (custom reader, section 6) |
| `cout << x << '\n';` | `out.println(x);` (`PrintWriter`) |
| `string out; out += to_string(x) + "\n"; cout << out;` | `StringBuilder sb = new StringBuilder(); sb.append(x).append('\n'); out.print(sb);` |
| `printf("%.10f\n", d);` | `out.printf("%.10f%n", d);` or `System.out.printf("%.10f%n", d);` |

## 5. Gotchas for C++ people

- **No references or pointers.** Objects and arrays are passed by reference, primitives by value. You can't write `void f(int& x)`.
- **Generics can't hold primitives.** `List<Integer>` boxes every element, which is slower and heavier on memory. Prefer `int[]` in hot loops.
- **Boxed comparison.** `list.get(i) == list.get(j)` compares references and only works by accident for values from -128 to 127. Use `.equals()` or unbox first.
- **`list.remove(int)` vs `list.remove(Object)`.** `list.remove(3)` removes the element at index 3, not the value 3.
- **Comparators.** Use `Integer.compare(a, b)` instead of `a - b`, which can overflow.
- **`Arrays.sort(int[])` can be anti-quicksort hacked** on Codeforces. Shuffle first, or sort an `Integer[]`.
- **Streams and `System.out.println` in a loop are slow.** Collect output and print once.
- **Default stack is small.** Deep recursion (DFS on 2x10^5 nodes) needs the big-stack thread trick or an iterative version.
- **Uninitialised arrays are zeroed** (`false` / `null` for other types), unlike C++ locals.

## 6. Input: reusable reader

Java's input is slow by default, so every pattern below runs on one reusable reader that pulls bytes in big chunks and parses them itself. Once you have it, each input format is just a few lines. In C++ `cin` already skips whitespace for you, and this reader does the same.

```java
import java.io.*;
import java.util.*;

public class Main {
    static class FastScanner {
        private final InputStream in = System.in;
        private final byte[] buf = new byte[1 << 16];
        private int ptr = 0, len = 0;

        private int read() throws IOException {
            if (ptr == len) {
                len = in.read(buf, 0, buf.length);
                ptr = 0;
                if (len <= 0) { len = 0; return -1; }   // EOF
            }
            return buf[ptr++];
        }
        int nextInt() throws IOException { return (int) nextLong(); }
        long nextLong() throws IOException {
            int b = read();
            while (b != -1 && b != '-' && (b < '0' || b > '9')) b = read();
            boolean neg = false;
            if (b == '-') { neg = true; b = read(); }
            long r = 0;
            while (b >= '0' && b <= '9') { r = r * 10 + (b - '0'); b = read(); }
            return neg ? -r : r;
        }
        double nextDouble() throws IOException { return Double.parseDouble(next()); }
        String next() throws IOException {            // one whitespace-separated token
            int b = read();
            while (b != -1 && b <= ' ') b = read();
            StringBuilder sb = new StringBuilder();
            while (b > ' ') { sb.append((char) b); b = read(); }
            return sb.toString();
        }
        char nextChar() throws IOException {
            int b = read();
            while (b != -1 && b <= ' ') b = read();
            return (char) b;
        }
        String nextLine() throws IOException {         // rest of the current line
            StringBuilder sb = new StringBuilder();
            int b = read();
            while (b != -1 && b != '\n') { if (b != '\r') sb.append((char) b); b = read(); }
            return sb.toString();
        }
        int[] nextIntArray(int n) throws IOException {
            int[] a = new int[n];
            for (int i = 0; i < n; i++) a[i] = nextInt();
            return a;
        }
        long[] nextLongArray(int n) throws IOException {
            long[] a = new long[n];
            for (int i = 0; i < n; i++) a[i] = nextLong();
            return a;
        }
    }

    public static void main(String[] args) throws IOException {
        FastScanner sc = new FastScanner();
        PrintWriter out = new PrintWriter(new BufferedWriter(new OutputStreamWriter(System.out)));
        // ... solve ...
        out.flush();
    }
}
```

The `throws IOException` is needed because reading can fail. Putting it on `main` means you never write try/catch.

## 7. Input: common formats

| Input format | C++ | Java (with `sc` above) |
|---|---|---|
| One int / long / double | `int x; long long y; double d; cin >> x >> y >> d;` | `int x = sc.nextInt(); long y = sc.nextLong(); double d = sc.nextDouble();` |
| One word | `string s; cin >> s;` | `String s = sc.next();` |
| One char | `char c; cin >> c;` | `char c = sc.nextChar();` |
| `n` then `n` numbers | `int n; cin >> n; vector<int> a(n); for (auto& x : a) cin >> x;` | `int n = sc.nextInt(); int[] a = sc.nextIntArray(n);` |
| `n x m` int matrix | `vector<vector<int>> g(n, vector<int>(m)); for (int i = 0; i < n; i++) for (int j = 0; j < m; j++) cin >> g[i][j];` | `int[][] g = new int[n][m]; for (int i = 0; i < n; i++) for (int j = 0; j < m; j++) g[i][j] = sc.nextInt();` |
| Grid of chars (rows like `#..#`) | `vector<string> g(n); for (auto& r : g) cin >> r;` | `char[][] g = new char[n][]; for (int i = 0; i < n; i++) g[i] = sc.next().toCharArray();` |
| Line with spaces | `string s; getline(cin, s);` | `String s = sc.nextLine();` |
| Edges, 1-indexed | `int a, b; cin >> a >> b; a--; b--;` | `int a = sc.nextInt() - 1, b = sc.nextInt() - 1;` |
| Multiple test cases | `int t; cin >> t; while (t--) { int n; cin >> n; }` | `int t = sc.nextInt(); while (t-- > 0) { int n = sc.nextInt(); }` |
| Queries `q` then `q` lines | `int q; cin >> q; while (q--) { int type, x; cin >> type >> x; }` | `int q = sc.nextInt(); while (q-- > 0) { int type = sc.nextInt(), x = sc.nextInt(); }` |
| Read until EOF | `int x; while (cin >> x) { }` | `String s; while ((s = br.readLine()) != null) { }` (`BufferedReader br`, section 9) |
| Pair of numbers | `pair<int,int> p; cin >> p.first >> p.second;` | `int[] p = {sc.nextInt(), sc.nextInt()};` |

**Mixing `nextLine` with other reads:** after `sc.nextInt()`, the newline is still unread, so a following `nextLine()` returns an empty string. Call `nextLine()` once to throw that away. C++ `getline` has the same trap after `cin >>`.

**Reading until EOF:** `nextInt` returns 0 at EOF, which can't be told apart from a real 0. If a problem says "until end of input", use the `BufferedReader` approach in section 9.

## 8. Input: graphs

```java
// Adjacency list, directed, n vertices, m edges
List<Integer>[] g = new ArrayList[n];
for (int i = 0; i < n; i++) g[i] = new ArrayList<>();
for (int i = 0; i < m; i++) {
    int a = sc.nextInt() - 1, b = sc.nextInt() - 1;
    g[a].add(b);
    // g[b].add(a);   // add this line for undirected
}

// Weighted: store {to, weight}
List<int[]>[] g = new ArrayList[n];
for (int i = 0; i < n; i++) g[i] = new ArrayList<>();
for (int i = 0; i < m; i++) {
    int a = sc.nextInt() - 1, b = sc.nextInt() - 1, w = sc.nextInt();
    g[a].add(new int[]{b, w});
}

// Tree with n-1 edges: same loop, run n-1 times
```

## 9. Input: `BufferedReader` fallback

Slower than the byte reader but simpler, and the right tool for EOF and line-oriented input.

```java
BufferedReader br = new BufferedReader(new InputStreamReader(System.in));

String line = br.readLine();                     // null at EOF
StringTokenizer st = new StringTokenizer(line);
int n = Integer.parseInt(st.nextToken());
int m = Integer.parseInt(st.nextToken());

// Read until EOF
String s;
while ((s = br.readLine()) != null) {
    // process one line
}

// Whole line as an int array
int[] a = Arrays.stream(br.readLine().trim().split(" ")).mapToInt(Integer::parseInt).toArray();
```

The `split(" ")` approach breaks if there are double spaces or trailing spaces. `StringTokenizer` handles those safely.

### Parsing structured lines with regex

For lines like `turn on 0,0 through 999,999`, a `Pattern` with capture groups beats `split` and `substring`. `import java.util.regex.*;`

```java
private static final Pattern INSTRUCTION =
        Pattern.compile("(turn on|turn off|toggle) (\\d+),(\\d+) through (\\d+),(\\d+)");

public void apply(String instruction) {
    Matcher m = INSTRUCTION.matcher(instruction);
    if (!m.matches()) throw new IllegalArgumentException(instruction);

    String action = m.group(1);
    int i1 = Integer.parseInt(m.group(2));
    int j1 = Integer.parseInt(m.group(3));
    int i2 = Integer.parseInt(m.group(4));
    int j2 = Integer.parseInt(m.group(5));

    switch (action) {
        case "turn on"  -> { /* ... */ }
        case "turn off" -> { /* ... */ }
        case "toggle"   -> { /* ... */ }
    }
}
```

Group 0 is the whole match and groups 1 onward are the parentheses in order. `matches()` requires the whole string to fit the pattern, while `find()` looks for a match anywhere inside it. Compile the `Pattern` once as a `static final`, not inside the per-line method. The arrow `case` syntax needs Java 14 or later, so use `case "toggle":` with `break;` on older judges.

## 10. Output

| C++ | Java |
|---|---|
| `cout << x << '\n';` | `out.println(x);` |
| `cout << a << ' ' << b << '\n';` | `out.println(a + " " + b);` |
| `for (int x : a) cout << x << ' '; cout << '\n';` (vector on one line) | `StringBuilder sb = new StringBuilder(); for (int x : a) sb.append(x).append(' '); out.println(sb.toString().trim());` |
| `for (int x : a) cout << x << '\n';` (one value per line, many lines) | `StringBuilder sb = new StringBuilder(); for (int x : a) sb.append(x).append('\n'); out.print(sb);` (print once at the end) |
| `printf("%.10f\n", d);` | `out.printf("%.10f%n", d);` |
| `cout << (ok ? "Yes" : "No") << '\n';` | `out.println(ok ? "Yes" : "No");` |

Always call `out.flush()` at the end, or nothing is printed.

## 11. Which reader to pick

- **Default:** the `FastScanner` above, which handles up to about 10^6 numbers comfortably.
- **Line-oriented or EOF-terminated input:** `BufferedReader`.
- **Interactive problems:** use `BufferedReader` and flush `System.out` after every query, since the byte-buffer reader may block waiting to fill its chunk.
- **Never use `Scanner`:** it can be 5-10x slower and causes time-limit failures at around 10^5 numbers.

## 12. Enums

```java
enum Dir { UP, DOWN, LEFT, RIGHT }

Dir d = Dir.UP;
if (d == Dir.UP) { /* compare with == */ }

switch (d) {
    case UP    -> { /* ... */ }      // no Dir. prefix inside case
    case DOWN  -> { /* ... */ }
    default    -> { /* ... */ }
}

for (Dir x : Dir.values()) { }       // iterate all constants
Dir e = Dir.valueOf("LEFT");         // string to enum, throws if no match
int idx = d.ordinal();               // 0-based position, handy as an array index
String name = d.name();              // "UP"
```

With data, for example direction deltas:

```java
enum Dir {
    UP(-1, 0), DOWN(1, 0), LEFT(0, -1), RIGHT(0, 1);
    final int dr, dc;
    Dir(int dr, int dc) { this.dr = dr; this.dc = dc; }
}
```

An enum is a class, so declare it at class level (inside `Main` as `enum Dir {...}`, or as a nested `static` type), not inside a method before Java 16. In C++ terms it behaves like `enum class`: constants are scoped under the type name and never convert to `int` implicitly. For plain integer codes in a hot loop, `static final int` constants are lighter.